In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
BRONZE_BASE = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net"
SILVER_BASE = "abfss://silver@stfinbankdevfbcq2026.dfs.core.windows.net"
ERRORES_PATH = f"{SILVER_BASE}/_errores/errores_pipeline"
 
try:
    filtro_tabla = dbutils.widgets.get("filtro_tabla")
except Exception:
    filtro_tabla = ""
 
try:
    run_id_actual = dbutils.widgets.get("run_id")
except Exception:
    run_id_actual = "manual"
 
def _debe_procesar(nombre_tabla_origen):
    return filtro_tabla == "" or filtro_tabla == nombre_tabla_origen
 
def particion_actual():
    from datetime import datetime, timezone
    hoy = datetime.now(timezone.utc)
    return hoy.strftime("%Y"), hoy.strftime("%m"), hoy.strftime("%d")
 
print(f"se ejecutar unicamente: '{filtro_tabla}'"
    if filtro_tabla
    else "se procesaran todas las tablas"
)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
 
schema_errores = StructType([
    StructField("tabla_origen", StringType(), False),
    StructField("id_registro_afectado", StringType(), True),
    StructField("motivo", StringType(), False),
    StructField("capa", StringType(), False),
    StructField("timestamp_error", TimestampType(), False),
])
 
def registrar_errores(df_errores, tabla_origen, motivo, id_col):
    if df_errores.count() == 0:
        return
    from pyspark.sql.functions import lit, current_timestamp, col
 
    df_para_log = df_errores.select(
        lit(tabla_origen).alias("tabla_origen"),
        col(id_col).cast("string").alias("id_registro_afectado"),
        lit(motivo).alias("motivo"),
        lit("Silver").alias("capa"),
        current_timestamp().alias("timestamp_error"),
    )
    df_para_log.write.format("delta").mode("append").save(ERRORES_PATH)
    print(f"  {df_errores.count()} registros con error ({motivo}) enviados a errores_pipeline")

In [0]:
def enmascarar_columnas(df, columnas_pii: list):
    from pyspark.sql.functions import sha2, col
    for columna in columnas_pii:
        df = df.withColumn(f"{columna}_hash", sha2(col(columna), 256)).drop(columna)
    return df

In [0]:
def validar_fk(df, columna_fk, df_referencia, columna_ref, nombre_tabla):
    ids_validos = df_referencia.select(columna_ref).distinct()
    df_validas = df.join(ids_validos, df[columna_fk] == ids_validos[columna_ref], "left_semi")
    df_invalidas = df.join(ids_validos, df[columna_fk] == ids_validos[columna_ref], "left_anti")
    if df_invalidas.count() > 0:
        registrar_errores(df_invalidas, nombre_tabla, f"FK inexistente: {columna_fk}", id_col=columna_fk)
    return df_validas

In [0]:
def upsert_silver(df_nuevo, silver_path, key_merge, nombre_tabla_log):
    from delta.tables import DeltaTable
 
    if DeltaTable.isDeltaTable(spark, silver_path):
        tabla_destino = DeltaTable.forPath(spark, silver_path)
        tabla_destino.alias("destino").merge(
            df_nuevo.alias("origen"),
            f"destino.{key_merge} = origen.{key_merge}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        print(f"MERGE aplicado sobre {nombre_tabla_log} (tabla ya existia)")
    else:
        df_nuevo.write.format("delta").mode("overwrite").save(silver_path)
        print(f"{nombre_tabla_log} creada por primera vez")
 
    total_final = spark.read.format("delta").load(silver_path).count()
    print(f"{nombre_tabla_log} tiene ahora {total_final} filas totales")

In [0]:
CALIDAD_PATH = f"{SILVER_BASE}/_calidad/reporte_calidad_silver"
 
def reporte_calidad(df_inicial, df_final, nombre_tabla, run_id_actual):
    import json
    from pyspark.sql import Row
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType
    from datetime import datetime, timezone
 
    total_inicial = df_inicial.count()
    total_final = df_final.count()
    rechazados = total_inicial - total_final
    pct_conformes = round((total_final / total_inicial * 100), 2) if total_inicial > 0 else 0.0
 
    nulos_por_columna = {}
    for columna in df_final.columns:
        nulos = df_final.filter(df_final[columna].isNull()).count()
        pct_nulo = round((nulos / total_final * 100), 2) if total_final > 0 else 0.0
        if pct_nulo > 0:
            nulos_por_columna[columna] = pct_nulo
 
    schema_calidad = StructType([
        StructField("nombre_tabla", StringType(), False),
        StructField("fecha_ejecucion", TimestampType(), False),
        StructField("total_filas_iniciales", IntegerType(), False),
        StructField("total_filas_finales", IntegerType(), False),
        StructField("registros_rechazados", IntegerType(), False),
        StructField("pct_conformes", DoubleType(), False),
        StructField("nulos_por_columna", StringType(), False),
        StructField("run_id", StringType(), False),
    ])
 
    fila = spark.createDataFrame(
        [Row(
            nombre_tabla=nombre_tabla,
            fecha_ejecucion=datetime.now(timezone.utc),
            total_filas_iniciales=total_inicial,
            total_filas_finales=total_final,
            registros_rechazados=rechazados,
            pct_conformes=pct_conformes,
            nulos_por_columna=json.dumps(nulos_por_columna),
            run_id=run_id_actual,
        )],
        schema=schema_calidad
    )
    fila.write.format("delta").mode("append").option("mergeSchema", "true").save(CALIDAD_PATH)
    print(f"reporte: {rechazados} rechazados, {pct_conformes}% conformes, nulos: {nulos_por_columna}")

In [0]:
def procesar_clientes():
    from pyspark.sql.functions import (
        col, floor, datediff, current_date, when, coalesce, lit,
        initcap, trim, concat_ws
    )

    year, month, day = particion_actual()
    bronze_path_hoy = f"{BRONZE_BASE}/TB_CLIENTES_CORE/{year}/{month}/{day}/"
 
    df_bronze = spark.read.parquet(bronze_path_hoy)
    print(f"Filas leidas de Bronze: {df_bronze.count()}")
 
    df_dedup = df_bronze.dropDuplicates(["id_cli"])
 
    df_sin_pk = df_dedup.filter(col("id_cli").isNull())
    if df_sin_pk.count() > 0:
        registrar_errores(df_sin_pk, "TB_CLIENTES_CORE", "id_cli nulo (registro descartado)", id_col="id_cli")
    df_dedup = df_dedup.filter(col("id_cli").isNotNull())
 
    df_nuevo = (
        df_dedup
        .withColumn("nombre_completo", concat_ws(" ", col("nomb_cli"), col("apell_cli")))
        .withColumn("edad", floor(datediff(current_date(), col("fec_nac")) / 365.25))
        .withColumn("score_buro_nulo", when(col("score_buro").isNull(), lit(1)).otherwise(lit(0)))
        .withColumn("depto_res", coalesce(col("depto_res"), lit("No informado")))
        .withColumn("ciudad_res", initcap(trim(col("ciudad_res"))))
        .withColumn("depto_res", initcap(trim(col("depto_res"))))
        .drop("nomb_cli", "apell_cli")
    )
    df_nuevo = enmascarar_columnas(df_nuevo, ["num_doc", "nombre_completo"])
    df_nuevo = df_nuevo.select(
        "id_cli", "nombre_completo_hash", "tip_doc", "num_doc_hash",
        "fec_nac", "edad", "fec_alta", "cod_segmento",
        "score_buro", "score_buro_nulo",
        "ciudad_res", "depto_res", "estado_cli", "canal_adquis",
    )
 
    silver_path = f"{SILVER_BASE}/clientes"
    reporte_calidad(df_bronze, df_nuevo, "TB_CLIENTES_CORE", run_id_actual)
    upsert_silver(df_nuevo, silver_path, "id_cli", "silver.clientes")
    return spark.read.format("delta").load(silver_path)
 
 
if _debe_procesar("TB_CLIENTES_CORE"):
    df_resultado = procesar_clientes()
    display(df_resultado.limit(10))
else:
    print("se omite esta tabla")

In [0]:
def procesar_productos():
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, desc, col
 
    df_bronze = spark.read.parquet(f"{BRONZE_BASE}/TB_PRODUCTOS_CAT/*/*/*/*.parquet")
    print(f"filas leidas de Bronze: {df_bronze.count()}")
 
    w = Window.partitionBy("cod_prod").orderBy(desc("ingestion_timestamp"))
    df_dedup = df_bronze.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")
    print(f"filas tras deduplicar por cod_prod: {df_dedup.count()}")
 
    df_sin_pk = df_dedup.filter(col("cod_prod").isNull())
    if df_sin_pk.count() > 0:
        registrar_errores(df_sin_pk, "TB_PRODUCTOS_CAT", "cod_prod nulo", id_col="cod_prod")
    df_dedup = df_dedup.filter(col("cod_prod").isNotNull())
 
    from pyspark.sql.functions import when, lit, pow as spark_pow
    df_silver = (
        df_dedup
        .withColumn("tasa_mensual_equiv", spark_pow(lit(1) + col("tasa_ea"), lit(1.0/12)) - lit(1))
        .withColumn(
            "familia_producto",
            when(col("tip_prod").contains("Credito"), "credito")
            .when(col("tip_prod").contains("Ahorro"), "ahorro")
            .otherwise("transaccional")
        )
    )
 
    duplicados = df_silver.groupBy("cod_prod").count().filter("count > 1").count()
    print(f"cod_prod: {duplicados} duplicados")
 
    silver_path = f"{SILVER_BASE}/productos"
    reporte_calidad(df_dedup, df_silver, "TB_PRODUCTOS_CAT", run_id_actual)
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)
    print(f"silver.productos escrito: {df_silver.count()} filas")
 
    return df_silver
 
 
if _debe_procesar("TB_PRODUCTOS_CAT"):
    df_productos = procesar_productos()
    display(df_productos.limit(10))
else:
    print("se omite esta tabla")

In [0]:
def procesar_sucursales():
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, desc, col
 
    df_bronze = spark.read.parquet(f"{BRONZE_BASE}/TB_SUCURSALES_RED/*/*/*/*.parquet")
    print(f"filas leidas de bronze: {df_bronze.count()}")
 
    w = Window.partitionBy("cod_suc").orderBy(desc("ingestion_timestamp"))
    df_dedup = df_bronze.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")
    print(f"Filas tras deduplicar por cod_suc: {df_dedup.count()}")
 
    df_sin_pk = df_dedup.filter(col("cod_suc").isNull())
    if df_sin_pk.count() > 0:
        registrar_errores(df_sin_pk, "TB_SUCURSALES_RED", "cod_suc nulo", id_col="cod_suc")
    df_silver = df_dedup.filter(col("cod_suc").isNotNull())
 
    duplicados = df_silver.groupBy("cod_suc").count().filter("count > 1").count()
    print(f"{duplicados} duplicados")
 
    silver_path = f"{SILVER_BASE}/sucursales"
    reporte_calidad(df_dedup, df_silver, "TB_SUCURSALES_RED", run_id_actual)
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_path)
    print(f"silver.sucursales escrito: {df_silver.count()} filas")
 
    return df_silver
 
 
if _debe_procesar("TB_SUCURSALES_RED"):
    df_sucursales = procesar_sucursales()
    display(df_sucursales.limit(10))
else:
    print("Se omite esta tabla")

In [0]:
def procesar_obligaciones():
    from pyspark.sql.functions import col
 
    year, month, day = particion_actual()
    bronze_path_hoy = f"{BRONZE_BASE}/TB_OBLIGACIONES/{year}/{month}/{day}/"
 
    df_bronze = spark.read.parquet(bronze_path_hoy)
    print(f"filas leidas de Bronze : {df_bronze.count()}")
 
    df_dedup = df_bronze.dropDuplicates(["id_oblig"])
 
    df_invalidas = df_dedup.filter((col("sdo_capital") < 0) | (col("dias_mora_act") < 0))
    if df_invalidas.count() > 0:
        registrar_errores(df_invalidas, "TB_OBLIGACIONES", "sdo_capital o dias_mora_act negativo", id_col="id_oblig")
    df_validas = df_dedup.filter((col("sdo_capital") >= 0) & (col("dias_mora_act") >= 0))
 
    df_silver_clientes = spark.read.format("delta").load(f"{SILVER_BASE}/clientes")
    df_silver_productos = spark.read.format("delta").load(f"{SILVER_BASE}/productos")
 
    df_validas = validar_fk(df_validas, "id_cli", df_silver_clientes, "id_cli", "TB_OBLIGACIONES")
    df_nuevo = validar_fk(df_validas, "cod_prod", df_silver_productos, "cod_prod", "TB_OBLIGACIONES")
 
    silver_path = f"{SILVER_BASE}/obligaciones"
    reporte_calidad(df_bronze, df_nuevo, "TB_OBLIGACIONES", run_id_actual)
    upsert_silver(df_nuevo, silver_path, "id_oblig", "silver.obligaciones")
    return spark.read.format("delta").load(silver_path)
 
 
if _debe_procesar("TB_OBLIGACIONES"):
    df_obligaciones = procesar_obligaciones()
    display(df_obligaciones.limit(10))
else:
    print("Se omite esta tabla")

In [0]:
def procesar_comisiones():
    year, month, day = particion_actual()
    bronze_path_hoy = f"{BRONZE_BASE}/TB_COMISIONES_LOG/{year}/{month}/{day}/"
 
    df_bronze = spark.read.parquet(bronze_path_hoy)
    print(f"filas leidas de bronze: {df_bronze.count()}")
 
    df_dedup = df_bronze.dropDuplicates(["id_comision"])
 
    df_silver_clientes = spark.read.format("delta").load(f"{SILVER_BASE}/clientes")
    df_silver_productos = spark.read.format("delta").load(f"{SILVER_BASE}/productos")
 
    df_validas = validar_fk(df_dedup, "id_cli", df_silver_clientes, "id_cli", "TB_COMISIONES_LOG")
    df_nuevo = validar_fk(df_validas, "cod_prod", df_silver_productos, "cod_prod", "TB_COMISIONES_LOG")
 
    silver_path = f"{SILVER_BASE}/comisiones"
    reporte_calidad(df_bronze, df_nuevo, "TB_COMISIONES_LOG", run_id_actual)
    upsert_silver(df_nuevo, silver_path, "id_comision", "silver.comisiones")
    return spark.read.format("delta").load(silver_path)
 
 
if _debe_procesar("TB_COMISIONES_LOG"):
    df_comisiones = procesar_comisiones()
    display(df_comisiones.limit(10))
else:
    print("Se omite esta tabla")

In [0]:
def procesar_movimientos():
    from pyspark.sql.functions import col, lit, current_date, to_date

    year, month, day = particion_actual()
    bronze_path_hoy = f"{BRONZE_BASE}/TB_MOV_FINANCIEROS/{year}/{month}/{day}/"

    df_bronze = spark.read.parquet(bronze_path_hoy)
    total_inicial = df_bronze.count()
    print(f"Filas leidas de Bronze: {total_inicial}")

    df_sin_dup = df_bronze.dropDuplicates(["id_mov"])
    duplicados_removidos = total_inicial - df_sin_dup.count()
    print(f"duplicados exactos removidos: {duplicados_removidos}")

    df_fechas_malas = df_sin_dup.filter(
        (to_date(col("fec_mov")) < to_date(lit("2015-01-01"))) |
        (to_date(col("fec_mov")) > current_date())
    )
    if df_fechas_malas.count() > 0:
        registrar_errores(df_fechas_malas, "TB_MOV_FINANCIEROS", "fecha fuera de rango valido", id_col="id_mov")
    df_fechas_ok = df_sin_dup.filter(
        (to_date(col("fec_mov")) >= to_date(lit("2015-01-01"))) &
        (to_date(col("fec_mov")) <= current_date())
    )
    print(f"fechas fuera de rango detectadas: {df_fechas_malas.count()}")

    df_montos_malos = df_fechas_ok.filter(col("vr_mov") <= 0)
    if df_montos_malos.count() > 0:
        registrar_errores(df_montos_malos, "TB_MOV_FINANCIEROS", "vr_mov invalido (<=0)", id_col="id_mov")
    df_montos_ok = df_fechas_ok.filter(col("vr_mov") > 0)
    print(f"montos invalidos detectados: {df_montos_malos.count()}")

    df_silver_clientes = spark.read.format("delta").load(f"{SILVER_BASE}/clientes")
    df_silver_productos = spark.read.format("delta").load(f"{SILVER_BASE}/productos")

    antes_fk = df_montos_ok.count()
    df_validas = validar_fk(df_montos_ok, "id_cli", df_silver_clientes, "id_cli", "TB_MOV_FINANCIEROS")
    print(f"id huerfanos (id_cli) detectados: {antes_fk - df_validas.count()}")

    df_silver_limpio = validar_fk(df_validas, "cod_prod", df_silver_productos, "cod_prod", "TB_MOV_FINANCIEROS")

    from pyspark.sql.window import Window
    from pyspark.sql.functions import unix_timestamp, avg, stddev, when, initcap, trim
    from delta.tables import DeltaTable

    silver_path_movimientos = f"{SILVER_BASE}/movimientos_financieros"
    if DeltaTable.isDeltaTable(spark, silver_path_movimientos):
        df_historico_ventana = (
            spark.read.format("delta").load(silver_path_movimientos)
            .select("id_mov", "id_cli", "fec_mov", "vr_mov")
        )
        df_hoy_ventana = df_silver_limpio.select("id_mov", "id_cli", "fec_mov", "vr_mov")
        df_para_ventana = df_historico_ventana.unionByName(df_hoy_ventana).dropDuplicates(["id_mov"])
        print(f"ventana movil: {df_historico_ventana.count()} historicas + {df_hoy_ventana.count()} de hoy")
    else:
        df_para_ventana = df_silver_limpio.select("id_mov", "id_cli", "fec_mov", "vr_mov")
        print("Sin historico previo, ventana movil calculada solo sobre filas de hoy")

    df_con_ts = df_para_ventana.withColumn(
        "fec_mov_unix", unix_timestamp(col("fec_mov").cast("timestamp"))
    )

    SEGUNDOS_POR_DIA = 86400
    ventana_30d = (
        Window.partitionBy("id_cli")
        .orderBy("fec_mov_unix")
        .rangeBetween(-30 * SEGUNDOS_POR_DIA, -1)
    )

    df_con_stats = (
        df_con_ts
        .withColumn("promedio_movil_30d", avg("vr_mov").over(ventana_30d))
        .withColumn("stddev_movil_30d", stddev("vr_mov").over(ventana_30d))
    )

    df_stats_calculadas = df_con_stats.withColumn(
        "ind_sospechoso",
        when(
            df_con_stats["promedio_movil_30d"].isNotNull()
            & df_con_stats["stddev_movil_30d"].isNotNull()
            & (col("vr_mov") > (col("promedio_movil_30d") + 3 * col("stddev_movil_30d"))),
            1
        ).otherwise(0)
    ).select("id_mov", "promedio_movil_30d", "stddev_movil_30d", "ind_sospechoso")

    df_con_flag = df_silver_limpio.join(df_stats_calculadas, on="id_mov", how="left")

    sospechosas = df_con_flag.filter(col("ind_sospechoso") == 1).count()
    print(f"transacciones marcadas como sospechosas: {sospechosas}")

    df_con_flag = df_con_flag.withColumn("cod_ciudad", initcap(trim(col("cod_ciudad"))))

    df_nuevo = enmascarar_columnas(df_con_flag, ["num_cuenta"])

    silver_path = f"{SILVER_BASE}/movimientos_financieros"
    reporte_calidad(df_bronze, df_nuevo, "TB_MOV_FINANCIEROS", run_id_actual)
    upsert_silver(df_nuevo, silver_path, "id_mov", "silver.movimientos_financieros")
    return spark.read.format("delta").load(silver_path)


if _debe_procesar("TB_MOV_FINANCIEROS"):
    df_movimientos = procesar_movimientos()
    display(df_movimientos.limit(10))
else:
    print("Se omite esta tabla")